1. 데이터 로드 및 구조 확인

In [ ]:
import seaborn as sns
iris = sns.load_dataset('iris')

In [ ]:
iris.head()

In [ ]:
iris.info()

2. 기술통계량

In [ ]:
# Species별 petal_length 평균
print(iris.groupby('species')['petal_length'].describe())

3. 시각화

In [ ]:
sns.boxplot(x='species', y='petal_length', data=iris)

setosa 종이 다른 종에 비해 petal_length가 짧고 분산이 더 적은 것을 확인할 수 있다.  
petal_length는 virginica, versicolor, setosa 순서대로 길다.

4. 정규성 검정 (Shapiro-Wilk)

#가설  
각 species들에 대해  
귀무가설(H0): {species}의 petal_length 데이터는 정규분포를 따른다.  
대립가설(H1): {species}의 petal_length 데이터는 정규분포를 따르지 않는다.

In [ ]:
from scipy import stats

for species in iris['species'].unique():
    data = iris[iris['species'] == species]['petal_length']
    stat, p = stats.shapiro(data)
    print(f"[{species}] p-value: {p:.4f}")

5. 등분산성 검정 (Levene)

In [ ]:
setosa = iris[iris['species'] == 'setosa']['petal_length']
versicolor = iris[iris['species'] == 'versicolor']['petal_length']
virginica = iris[iris['species'] == 'virginica']['petal_length']

stat, p = stats.levene(setosa, versicolor, virginica)
print(f"Levene's test p-value: {p: .4f}")

6. ANOVA 가설 수립

#가설   
귀무가설(H0): 세 species 간 petal_length의 평균은 모두 같다.  
대립가설(H1): 적어도 하나의 species의 petal_length 평균은 다르다.

7. One-way ANOVA

In [ ]:
setosa = iris[iris['species'] == 'setosa']['petal_length']
versicolor = iris[iris['species'] == 'versicolor']['petal_length']
virginica = iris[iris['species'] == 'virginica']['petal_length']

f, p = stats.f_oneway(setosa, versicolor, virginica)

print(f"ANOVA F-statistic: {f: .4f}, p-value: {p: .4f}")

p값이 0.05 미만이므로 귀무가설을 기각한다.  
따라서 세 species의 petal_length 평균은 차이가 있다고 할 수 있다.

8. 사후검정 (Tukey HSD)

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(endog=iris['petal_length'],
                          groups=iris['species'],
                          alpha=0.05)

print(tukey)

세 species 모두 petal_length 평균 간 유의미한 차이가 있음을 알 수 있다.

9. 결과 요약

Boxplot, ANOVA, 사후검정 결과를 종합하였을 때  
petal_length는 virginica, versicolor, setosa 순서대로 통계적으로 유의하게 길다.

10. 회귀 분석

In [ ]:
from sklearn.linear_model import LinearRegression